In [1]:
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, f1_score

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

In [2]:
DATA_PATH = Path("/home/chupchik/Рабочий стол/fisrt_stepD/The_final_recomendation/output_dataset/ait_ads_wazuh_raw.parquet")

df = pd.read_parquet(DATA_PATH)

print(df.shape)
df.head()

(2493428, 15)


,scenario,timestamp,agent_id,agent_name,agent_ip,hostname,program,location,full_log,rule_id,rule_level,rule_description,rule_groups,rule_groups_str,y_priority
0,fox,2022-01-15 02:32:32+00:00,18,wazuh-client,172.17.131.81,mail,freshclam,/var/log/syslog,Jan 15 02:32:32 mail freshclam[29266]: Sat Jan...,52507,3,ClamAV database update,"[clamd, freshclam, virus]","clamd,freshclam,virus",low
1,fox,2022-01-15 02:32:32+00:00,6,wazuh-client,192.168.128.170,taylorcruz-mail,freshclam,/var/log/syslog,Jan 15 02:32:32 taylorcruz-mail freshclam[2851...,52507,3,ClamAV database update,"[clamd, freshclam, virus]","clamd,freshclam,virus",low
2,fox,2022-01-15 02:32:37+00:00,18,wazuh-client,172.17.131.81,mail,freshclam,/var/log/syslog,Jan 15 02:32:37 mail freshclam[29266]: Sat Jan...,52507,3,ClamAV database update,"[clamd, freshclam, virus]","clamd,freshclam,virus",low
3,fox,2022-01-15 02:32:42+00:00,18,wazuh-client,172.17.131.81,mail,freshclam,/var/log/syslog,Jan 15 02:32:42 mail freshclam[29266]: Sat Jan...,52507,3,ClamAV database update,"[clamd, freshclam, virus]","clamd,freshclam,virus",low
4,fox,2022-01-15 02:32:47+00:00,18,wazuh-client,172.17.131.81,mail,freshclam,/var/log/syslog,Jan 15 02:32:47 mail freshclam[29266]: Sat Jan...,52507,3,ClamAV database update,"[clamd, freshclam, virus]","clamd,freshclam,virus",low


In [3]:
target_col = "y_priority"

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df[target_col])

print(label_encoder.classes_)
print(df[target_col].value_counts())

['critical' 'high' 'low' 'medium']
y_priority
medium      1873575
low          486285
high         131901
critical       1667
Name: count, dtype: int64


In [4]:
def run_text_test(df, text_col, y, label_encoder, test_size=0.2, random_state=42):
    tmp = df[[text_col]].copy()
    tmp[text_col] = tmp[text_col].fillna("")

    X_train, X_test, y_train, y_test = train_test_split(
        tmp[text_col],
        y,
        test_size=test_size,
        random_state=random_state,
        stratify=y
    )

    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(max_features=3000, ngram_range=(1, 2))),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    print(f"\n===== ТЕСТ ПРИЗНАКА: {text_col} =====")
    print("Macro F1:", f1_score(y_test, y_pred, average="macro"))
    print(classification_report(
        y_test,
        y_pred,
        target_names=label_encoder.classes_,
        digits=4
    ))

In [5]:
run_text_test(df, "rule_description", y, label_encoder)


===== ТЕСТ ПРИЗНАКА: rule_description =====
Macro F1: 0.9999442145443285
              precision    recall  f1-score   support

    critical     1.0000    1.0000    1.0000       334
        high     1.0000    0.9998    0.9999     26380
         low     0.9998    1.0000    0.9999     97257
      medium     1.0000    1.0000    1.0000    374715

    accuracy                         1.0000    498686
   macro avg     1.0000    0.9999    0.9999    498686
weighted avg     1.0000    1.0000    1.0000    498686



In [6]:
run_text_test(df, "rule_groups_str", y, label_encoder)


===== ТЕСТ ПРИЗНАКА: rule_groups_str =====
Macro F1: 0.7218596052815509
              precision    recall  f1-score   support

    critical     0.0055    1.0000    0.0109       334
        high     0.9996    0.9309    0.9640     26380
         low     0.9961    0.9998    0.9980     97257
      medium     0.9999    0.8425    0.9145    374715

    accuracy                         0.8780    498686
   macro avg     0.7503    0.9433    0.7219    498686
weighted avg     0.9985    0.8780    0.9328    498686



In [7]:
run_text_test(df, "full_log", y, label_encoder)


===== ТЕСТ ПРИЗНАКА: full_log =====
Macro F1: 0.35444231008970944
              precision    recall  f1-score   support

    critical     0.0076    0.7455    0.0151       334
        high     0.0704    0.8144    0.1296     26380
         low     0.9997    1.0000    0.9999     97257
      medium     0.9419    0.1598    0.2732    374715

    accuracy                         0.3587    498686
   macro avg     0.5049    0.6799    0.3544    498686
weighted avg     0.9064    0.3587    0.4072    498686

